# Descriptive statistics

Templates, formulas and narrated (LLM-filled) data now live in the `simulator` package.
This notebook only shows how to use it. CLI equivalent: `python -m simulator --help`.

In [ ]:
import pandas as pd

from simulator import formulas as fm
from simulator.narrated import Operation, apply_operations, missing_report, show
from simulator.scenario import Scenario
from simulator.templates import KINDS, load_template

scenario = Scenario.open("world-1990-2025")

# Templates

Developer-defined schemas (`simulator/common/templates/<kind>.csv`), one row per field.

In [ ]:
for kind in KINDS:
    display(kind, load_template(kind).fields)

# Formulas

User-editable rows (`simulator/scenarios/<scenario>/formulas/<kind>.csv`). Edit them with `fm.add_formula`, `fm.edit_formula`, `fm.remove_formula` (they migrate narrated data).

In [ ]:
for kind in KINDS:
    display(kind, fm.load_table(scenario, kind))

# Setting narrated values

One operation = one formula, targets, closed year interval and a change curve (`linear`, `exponential`, `logistic`, `step`, `constant`, `manual`). `dry_run=True` writes nothing.

In [ ]:
results = apply_operations(
    scenario,
    [
        Operation(kind="grade", name="stability", targets="POL,UKR", years="1990-2000",
                  curve="logistic", start=40, end=70),
        Operation(kind="directed", name="diplomatic relation towards", targets="POL:RUS",
                  years="2014-2022", curve="exponential", start=0, end=-80, params={"k": 2}),
        Operation(kind="doublesided", name="in war", targets="RUS:UKR", years="2014-2025",
                  curve="constant", start=1),
    ],
    author="notebook",
    rationale="example",
    dry_run=True,
)
for r in results:
    display(r.formula.name, r.preview)

# Reading values and coverage

In [ ]:
display(show(scenario, "grade", "stability", "POL,RUS", "1990-1995"))
missing_report(scenario)